# SMS AC interactive project discussion
Run this notebook interactively on Kaggle with GPU and Internet enabled. Attach the current project dataset and the completed training notebook outputs. This notebook never trains or uses a laptop GPU. Execute the discussion cell repeatedly, then the report cell. Model loading occurs once per command; this simple isolated process interface prioritizes reproducibility over latency. Do not Save and Run All: the discussion needs your input.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, zipfile

assert platform.system() == "Linux" and Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle."
assert any(os.environ.get(k) for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_DATA_PROXY_TOKEN")), "Kaggle environment marker is missing."

PROJECT = Path("/kaggle/working/SEproject")
RUN_FULL = False           # True trains every eligible training row for FULL_EPOCHS.
PILOT_STEPS = 150          # <=5 is recorded as smoke; >5 is pilot.
FULL_EPOCHS = 1
MAX_SEQ_LENGTH = 4096      # Complete prompt + answer; oversized rows fail, never truncate.
GRADIENT_ACCUMULATION = 8
RESUME = {"design": None, "sdlc": None}  # "auto" or checkpoint directory after an interrupted run.
TEST_LIMIT = 12            # Set 0 for all held-out examples.
GENERATION_LIMIT = 3
CHALLENGE_LIMIT = 1        # Independent prompts, not expert-validated gold answers.
MAX_NEW_TOKENS = 3072
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BASE_REVISION = "main"     # Each run resolves this to an immutable commit and records it.

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["SMS_AC_GPU"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"  # Keep subprocess metrics visible during long runs.
os.environ["HF_HOME"] = "/tmp/sms-ac-huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
print("Run tier:", "full" if RUN_FULL else ("smoke" if PILOT_STEPS <= 5 else "pilot"))


In [ ]:
# Kaggle's browser uploader may expand ZIPs into an SEproject directory.
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    unpacked = [p.parent.parent for p in Path("/kaggle/input").rglob("scripts/train_dual_qlora.py")]
    if len(unpacked) > 1:
        raise RuntimeError(f"Attach exactly one project dataset: {unpacked}")
    if unpacked:
        import shutil
        shutil.copytree(unpacked[0], PROJECT, dirs_exist_ok=True)
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    accepted = {"project.zip", "seproject.zip", "sms-ac-project.zip"}
    archives = [p for p in Path("/kaggle/input").rglob("*.zip") if p.name.lower() in accepted]
    if len(archives) != 1:
        raise RuntimeError(f"Attach exactly one project.zip dataset. Candidates: {archives}")
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        entries = archive.infolist()
        if sum(item.file_size for item in entries) > 3 * 1024**3:
            raise ValueError("Project archive exceeds the 3 GiB extraction budget.")
        mapped = []
        seen = set()
        for item in entries:
            normalized = item.filename.replace("\\", "/")
            parts = Path(normalized).parts
            if parts and parts[0].lower() == "seproject":
                parts = parts[1:]
            if not parts:
                continue
            if normalized.startswith("/") or ".." in parts or any(":" in part for part in parts):
                raise ValueError(f"Unsafe archive path: {item.filename}")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not allowed.")
            target = PROJECT.joinpath(*parts).resolve()
            if not target.is_relative_to(PROJECT.resolve()) or target in seen:
                raise ValueError(f"Unsafe or duplicate target: {target}")
            seen.add(target)
            mapped.append((item, target))
        for item, target in mapped:
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("wb") as destination:
                    import shutil
                    shutil.copyfileobj(source, destination)
assert (PROJECT / "requirements-kaggle.txt").exists()
for task in ("design", "sdlc"):
    for split in ("train", "validation", "test"):
        assert (PROJECT / "data/dual" / task / f"{split}.jsonl").is_file(), f"Missing {task}/{split}"
print("Project:", PROJECT)


In [ ]:
import venv
VENV = Path("/tmp/sms-ac-training-venv")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(VENV)
# Kaggle may omit ensurepip; host pip can bootstrap an empty venv.
subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON),
                "install", "pip==25.3"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-kaggle.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
(PROJECT / "reports").mkdir(exist_ok=True)
lock = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(PROJECT / "reports/kaggle-environment.txt").write_text(lock)
print("Isolated training environment ready.")


In [ ]:
import shutil
from pathlib import Path
for task in ('design','sdlc'):
    target=PROJECT/'models'/f'{task}-adapter'
    if not (target/'model_manifest.json').exists():
        candidates=[p.parent for p in Path('/kaggle/input').rglob('model_manifest.json') if p.parent.name==f'{task}-adapter']
        if len(candidates)!=1:
            raise RuntimeError(f'Attach exactly one completed {task} adapter output: {candidates}')
        shutil.copytree(candidates[0],target,dirs_exist_ok=True)
SESSION=PROJECT/'outputs/interactive_conversation.json'
SESSION.parent.mkdir(parents=True,exist_ok=True)
conversation=json.loads(SESSION.read_text()) if SESSION.exists() else []
print('Loaded discussion turns:',len(conversation))


## Discussion
Run the following cell once per answer. Conversation text stays in the working output folder.

In [ ]:
answer=input('Describe your networking project or answer the latest question: ').strip()
if not answer: raise ValueError('Enter a project description or answer')
if conversation and conversation[-1]['role']=='user': conversation[-1]['content']+='\n'+answer
else: conversation.append({'role':'user','content':answer})
SESSION.write_text(json.dumps(conversation,ensure_ascii=False,indent=2))
command=[str(PYTHON),str(PROJECT/'scripts/run_sms_ac.py'),'clarify','--conversation',str(SESSION),'--models',str(PROJECT/'models')]
result=subprocess.run(command,cwd=PROJECT,text=True,capture_output=True)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('Clarification failed; your answer was saved for retry')
reply=result.stdout.strip()
conversation.append({'role':'assistant','content':reply})
SESSION.write_text(json.dumps(conversation,ensure_ascii=False,indent=2))
print(reply)


## Generate both Word reports
Run after discussion. Set PROVISIONAL to True only when you want a draft with unresolved requirements.

In [ ]:
PROVISIONAL=False
if not conversation: raise ValueError('Discuss your project first')
output=PROJECT/'outputs/my-project'
command=[str(PYTHON),str(PROJECT/'scripts/run_sms_ac.py'),'all','--request',conversation[0]['content'],
         '--conversation',str(SESSION),'--models',str(PROJECT/'models'),'--output-dir',str(output)]
if PROVISIONAL: command.append('--provisional')
subprocess.run(command,cwd=PROJECT,check=True)
from IPython.display import FileLink,display
for path in output.glob('*.docx'): display(FileLink(str(path)))


## Analyze another Word document
Upload it as a private notebook input and set DOCUMENT_PATH to its visible Kaggle path.

In [ ]:
DOCUMENT_PATH=''  # Explicit path to your uploaded .docx
if DOCUMENT_PATH:
    subprocess.run([str(PYTHON),str(PROJECT/'scripts/run_sms_ac.py'),'sdlc','--docx',DOCUMENT_PATH,
                    '--models',str(PROJECT/'models'),'--output-dir',str(PROJECT/'outputs/uploaded-document')],cwd=PROJECT,check=True)
    display(FileLink(str(PROJECT/'outputs/uploaded-document/sdlc_recommendation.docx')))
